In [ ]:
# Find it, understand it, fix it. That's mechanistic interpretability.

In [ ]:
# [Cell 1] Install dependencies
# !pip install transformer_lens sae_lens transformers torch
# !pip install --force-reinstall numpy scipy pandas scikit-learn

# If on Colab, restart runtime after install:
# Runtime -> Restart runtime (Ctrl+M+.), then run from Cell 2

## Setup
Load GPT-2 and SAE weights.

In [1]:
# [Cell 2] Load GPT-2 model
import torch
import transformer_lens as tl
import os
# from google.colab import drive

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device}')

# --- Colab (Google Drive) ---
# drive.mount('/content/drive')
# save_path = '/content/drive/MyDrive/gpt2_small_hooked.pt'

# --- Local ---
save_dir = os.path.dirname(os.path.abspath('__file__'))
save_path = os.path.join(save_dir, 'gpt2_small_hooked.pt')

if os.path.exists(save_path):
    model = tl.HookedTransformer.from_pretrained('gpt2-small', device='cpu')
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.to(device)
    print("Loaded from local cache")
else:
    model = tl.HookedTransformer.from_pretrained('gpt2-small', device=device)
    torch.save(model.state_dict(), save_path)
    print(f"Downloaded and saved to {save_path}")

print(f'GPT-2 small: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params')
print(f'dim={model.cfg.d_model}, layers={model.cfg.n_layers}, heads={model.cfg.n_heads}')

Using: cpu


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
Moving model to device:  cpu
Loaded from local cache
GPT-2 small: 163M params
dim=768, layers=12, heads=12


In [3]:
# [Cell 3] Load SAE as raw tensors
# sae_lens downloads the weights, then we extract 4 tensors and only use those
from sae_lens import SAE

_sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.8.hook_resid_pre",
    device=device
)

# Extract the 4 tensors that define the entire SAE
W_enc = _sae.W_enc.detach()   # [768, 24576]  encoder weights
b_enc = _sae.b_enc.detach()   # [24576]       encoder bias
W_dec = _sae.W_dec.detach()   # [24576, 768]  decoder weights
b_dec = _sae.b_dec.detach()   # [768]         decoder bias
del _sae

# SAE encode = ReLU((x - b_dec) @ W_enc + b_enc)
# SAE decode = features @ W_dec + b_dec
print(f"W_enc: {W_enc.shape}  (768 -> 24576)")
print(f"b_enc: {b_enc.shape}")
print(f"W_dec: {W_dec.shape}  (24576 -> 768)")
print(f"b_dec: {b_dec.shape}")

W_enc: torch.Size([768, 24576])  (768 -> 24576)
b_enc: torch.Size([24576])
W_dec: torch.Size([24576, 768])  (24576 -> 768)
b_dec: torch.Size([768])


/home/cody/anaconda3/lib/python3.12/site-packages/sae_lens/saes/sae.py:251: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


## Single Prompt
Run one prompt through GPT-2, encode residual stream through SAE, see which features fire per token.

In [ ]:
# [Cell 4] SAE encode = matmul + ReLU
# Run prompt through GPT-2, grab layer 8 residual stream, encode through SAE

prompt = "The capital of France is"
_, cache = model.run_with_cache(prompt) ############ keep going
x = cache["blocks.8.hook_resid_pre"]  # GPT-2 residual: [1, seq_len, 768]

# SAE encode: features = ReLU((x - b_dec) @ W_enc + b_enc)
features = torch.relu((x - b_dec) @ W_enc + b_enc)  # [1, seq_len, 24576]

print(f"GPT-2 residual shape: {x.shape}")
print(f"SAE features shape:   {features.shape}")

# How many SAE features fire per token?
tokens = model.to_str_tokens(prompt)
for i, tok in enumerate(tokens):
    n_active = (features[0, i] > 0).sum().item()
    print(f"  Token '{tok}': {n_active} active features out of {features.shape[-1]}")

GPT-2 residual shape: torch.Size([1, 6, 768])
SAE features shape:   torch.Size([1, 6, 24576])
  Token '<|endoftext|>': 8 active features out of 24576
  Token 'The': 8 active features out of 24576
  Token ' capital': 35 active features out of 24576
  Token ' of': 66 active features out of 24576
  Token ' France': 47 active features out of 24576
  Token ' is': 51 active features out of 24576


In [6]:
# [Cell 5] Top SAE features for last token
last_acts = features[0, -1]  # [24576]
top = torch.topk(last_acts, 10)

print(f"Top 10 SAE features on last token '{tokens[-1]}':")
print(f"{'Feature ID':>12} | {'Activation':>10}")
print("-" * 27)
for fid, val in zip(top.indices, top.values):
    print(f"{fid.item():>12} | {val.item():>10.3f}")

Top 10 SAE features on last token ' is':
  Feature ID | Activation
---------------------------
       21000 |     10.298
       18220 |      8.410
       14430 |      7.472
        1442 |      6.797
       19805 |      6.314
       21496 |      6.313
        6863 |      6.239
       12013 |      5.085
       19182 |      4.214
       15547 |      3.901


## Steering
Add SAE decoder direction to GPT-2's residual stream to shift output toward a concept.

In [7]:
# [Cell 6] Steering = extract one row from W_dec, add to GPT-2 residual stream
FEATURE_ID = top.indices[0].item()
COEFF = 10.0

# Steering vector = one row of SAE decoder matrix, scaled
steer = COEFF * W_dec[FEATURE_ID]  # [768]

def steering_hook(resid, hook):
    resid[:, :, :] += steer
    return resid

prompt = "I think the best city in the world is"
print("=== Without steering ===")
print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

print(f"\n=== With steering (SAE feature {FEATURE_ID}, coeff={COEFF}) ===")
with model.hooks(fwd_hooks=[("blocks.8.hook_resid_pre", steering_hook)]):
    print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

=== Without steering ===


  0%|          | 0/50 [00:00<?, ?it/s]

I think the best city in the world is LA, because it is so amazing. LA has been growing since I was a kid. I have seen my city grow and I am proud of it. I am very proud of that. If I were to move to LA I'd be one of

=== With steering (SAE feature 21000, coeff=10.0) ===


  0%|          | 0/50 [00:00<?, ?it/s]

I think the best city in the world is probably not New York City. I think it almost looks like Chicago, but I don't think it's actually that big of a deal. Well, it's interesting to look at New York City. It's a major hub for commerce, and you


## Feature Search
Run multiple prompts to find which SAE features correspond to a concept, then verify what a feature represents.

In [ ]:
# [Cell 7] Concept -> feature search
from collections import Counter

CONCEPT = "Golden Gate Bridge"
search_prompts = [
    f"The {CONCEPT} looks cool",
    f"The meaning of {CONCEPT} is",
    f"{CONCEPT} is something that everyone",
    f"The landmark {CONCEPT}",
    f"People often associate {CONCEPT} with",
]

concept_words = [w.lower() for w in CONCEPT.split()]
counts = Counter()
max_acts = {}

for p in search_prompts:
    toks = model.to_str_tokens(p)
    _, c = model.run_with_cache(p)
    x = c["blocks.8.hook_resid_pre"]
    feats = torch.relu((x - b_dec) @ W_enc + b_enc)
    for i, tok in enumerate(toks):
        if any(w in tok.strip().lower() for w in concept_words):
            top10 = torch.topk(feats[0, i], 10)
            for fid, val in zip(top10.indices.tolist(), top10.values.tolist()):
                counts[fid] += 1
                max_acts[fid] = max(max_acts.get(fid, 0), val)

print(f"SAE features for \"{CONCEPT}\":\n")
print(f"{'Feature ID':>10} | {'Times seen':>10} | {'Max activation':>14}")
print("-" * 42)
for fid, cnt in counts.most_common(15):
    print(f"{fid:>10} | {cnt:>10} | {max_acts[fid]:>14.3f}")

SAE features for "Golden Gate Bridge":

Feature ID | Times seen | Max activation
------------------------------------------
      7525 |         10 |         11.706
       735 |          8 |         12.790
     10261 |          5 |         75.465
      7451 |          5 |          6.995
      8198 |          5 |         27.872
     17840 |          5 |         16.332
     23937 |          5 |         10.786
      3808 |          5 |          7.535
       990 |          5 |          6.821
     17608 |          5 |         41.628
     16031 |          5 |          8.430
      5055 |          5 |          7.304
      6863 |          5 |          4.776
      4587 |          4 |         13.108
     21665 |          4 |          6.182


In [17]:
# [Cell 8] Feature -> concept lookup
# Given a feature ID, find what concept it represents
FEATURE_ID = 10261  # try 974 (Paris), 10261 (Golden), 17608 (Bridge), etc.

probe_prompts = [
    "The Golden Gate Bridge spans San Francisco Bay",
    "San Francisco is in California",
    "The Brooklyn Bridge connects Manhattan and Brooklyn",
    "Bridges are built over rivers and bays",
    "The Bay Area has many tech companies",
    "The Golden Gate Bridge is huge",
]

results = []
for p in probe_prompts:
    toks = model.to_str_tokens(p)
    _, c = model.run_with_cache(p)
    x = c["blocks.8.hook_resid_pre"]
    feats = torch.relu((x - b_dec) @ W_enc + b_enc)
    for i, tok in enumerate(toks):
        val = feats[0, i, FEATURE_ID].item()
        if val > 0:
            results.append((val, tok, p))

results.sort(reverse=True)

print(f"Feature {FEATURE_ID} fires on:\n")
for val, tok, prompt in results[:15]:
    print(f"  {val:>8.3f}  '{tok}'  <- {prompt}")

Feature 10261 fires on:

    73.706  ' Golden'  <- The Golden Gate Bridge spans San Francisco Bay


## Multi-Feature Steering
Combine multiple SAE feature directions to steer toward a complex concept.

In [22]:
# [Cell 9] Golden Gate Bridge steering — combine multiple SAE feature directions
gg_features = {
    10261: 40.0,  # "Gate"
    7525: 300
}

# Steering vector = sum of scaled SAE decoder rows
steer = sum(coeff * W_dec[fid] for fid, coeff in gg_features.items())  # [768]

def gg_hook(resid, hook):
    resid[:, :, :] += steer
    return resid

prompt = "I think the best place to visit is"
print("=== Without steering ===")
print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

print(f"\n=== With Golden Gate Bridge steering ===")
with model.hooks(fwd_hooks=[("blocks.8.hook_resid_pre", gg_hook)]):
    print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

=== Without steering ===


  0%|          | 0/50 [00:00<?, ?it/s]

I think the best place to visit is in Africa. I can't live in this country without cycling. I've been cycling for 15 years.

I've been cycling for years and I've loved doing it. I've gone to all the races, I've been at the water

=== With Golden Gate Bridge steering ===


  0%|          | 0/50 [00:00<?, ?it/s]

I think the best place to visit is, City City,,, City-,,,-,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


## Circuit Tracing
Ablate one attention head at a time (partial derivative) to find which heads cause a feature to fire.

In [ ]:
# [Cell 10] Circuit tracing — ablate attention heads AND MLPs (unigram feature)
# Partial derivative: zero one component, measure effect on SAE feature

TARGET_FEATURE = 10261  # Golden Gate feature from Cell 7
prompt = "The Golden Gate Bridge spans"

# Step 1: baseline activations per token
_, baseline_cache = model.run_with_cache(prompt)
baseline_x = baseline_cache["blocks.8.hook_resid_pre"]
baseline_acts = torch.relu((baseline_x - b_dec) @ W_enc + b_enc)

tokens = model.to_str_tokens(prompt)
token_acts = baseline_acts[0, :, TARGET_FEATURE]
best_tok = token_acts.argmax().item()
baseline_val = token_acts[best_tok].item()

print(f"Feature {TARGET_FEATURE} activations per token:")
for i, tok in enumerate(tokens):
    print(f"  '{tok}': {token_acts[i].item():.3f}")
print(f"\nTracing token {best_tok} '{tokens[best_tok]}' (activation = {baseline_val:.3f})")

# Step 2: ablate each component one at a time
results = []

# Ablate attention heads (layers 0-7) via hook_z [batch, pos, heads, d_head=64]
for layer in range(8):
    for head in range(model.cfg.n_heads):
        def ablate_head(value, hook, head_idx=head):
            value[:, :, head_idx, :] = 0
            return value
        with model.hooks(fwd_hooks=[(f"blocks.{layer}.attn.hook_z", ablate_head)]):
            _, abl_cache = model.run_with_cache(prompt)
        abl_x = abl_cache["blocks.8.hook_resid_pre"]
        abl_acts = torch.relu((abl_x - b_dec) @ W_enc + b_enc)
        abl_val = abl_acts[0, best_tok, TARGET_FEATURE].item()
        change = abl_val - baseline_val
        results.append((f"L{layer} Attn H{head}", change))

# Ablate MLP layers (layers 0-7)
for layer in range(8):
    def ablate_mlp(value, hook):
        value[:, :, :] = 0
        return value
    with model.hooks(fwd_hooks=[(f"blocks.{layer}.hook_mlp_out", ablate_mlp)]):
        _, abl_cache = model.run_with_cache(prompt)
    abl_x = abl_cache["blocks.8.hook_resid_pre"]
    abl_acts = torch.relu((abl_x - b_dec) @ W_enc + b_enc)
    abl_val = abl_acts[0, best_tok, TARGET_FEATURE].item()
    change = abl_val - baseline_val
    results.append((f"L{layer} MLP", change))

# Sort by absolute impact
results.sort(key=lambda r: abs(r[1]), reverse=True)
max_change = max(abs(r[1]) for r in results) if results else 1

print(f"\n{'Component':>15} | {'Change':>10} | Impact")
print("-" * 55)
for name, change in results[:20]:
    if abs(change) < 0.001:
        continue
    bar = "█" * int(abs(change) / max_change * 20) if max_change > 0 else ""
    direction = "▼" if change < 0 else "▲"
    print(f"  {name:>13} | {change:>+10.3f} | {direction} {bar}")

if all(abs(r[1]) < 0.001 for r in results):
    print("  No component has significant impact — feature likely comes from embedding alone")

In [32]:
# [Cell 11] Contextual circuit trace — IOI task, measuring logit for "Mary"
# Fixed: use hook_z [batch, pos, heads, d_head=64] instead of hook_result

full_prompt = "John and Mary went to the store. John gave a drink to"
mary_token = model.to_tokens(" Mary")[0, 1]

baseline_logits = model(full_prompt)
baseline_logit = baseline_logits[0, -1, mary_token].item()

top_preds = torch.topk(baseline_logits[0, -1], 5)
print("Top 5 predictions after '...gave a drink to':")
for tid, val in zip(top_preds.indices, top_preds.values):
    print(f"  '{model.to_string(tid)}': {val.item():.3f}")
print(f"\nBaseline logit for ' Mary': {baseline_logit:.3f}")

# Ablate each attention head (via hook_z) and MLP
results = []
for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):
        def ablate_head(value, hook, head_idx=head):
            value[:, :, head_idx, :] = 0  # zero head in hook_z [batch, pos, head, d_head=64]
            return value
        abl_logits = model.run_with_hooks(
            full_prompt,
            fwd_hooks=[(f"blocks.{layer}.attn.hook_z", ablate_head)]
        )
        abl_logit = abl_logits[0, -1, mary_token].item()
        results.append((f"L{layer} Attn H{head}", abl_logit - baseline_logit))

for layer in range(model.cfg.n_layers):
    def ablate_mlp(value, hook):
        value[:, :, :] = 0
        return value
    abl_logits = model.run_with_hooks(
        full_prompt,
        fwd_hooks=[(f"blocks.{layer}.hook_mlp_out", ablate_mlp)]
    )
    abl_logit = abl_logits[0, -1, mary_token].item()
    results.append((f"L{layer} MLP", abl_logit - baseline_logit))

results.sort(key=lambda r: abs(r[1]), reverse=True)
max_change = max(abs(r[1]) for r in results) if results else 1

print(f"\n{'Component':>15} | {'Δ logit(Mary)':>13} | Impact")
print("-" * 60)
for name, change in results[:20]:
    if abs(change) < 0.01:
        continue
    bar = "█" * int(abs(change) / max_change * 20) if max_change > 0 else ""
    direction = "▼" if change < 0 else "▲"
    print(f"  {name:>13} | {change:>+13.3f} | {direction} {bar}")

Top 5 predictions after '...gave a drink to':
  ' Mary': 18.106
  ' them': 16.306
  ' the': 16.019
  ' John': 15.798
  ' his': 15.387

Baseline logit for ' Mary': 18.106

      Component | Δ logit(Mary) | Impact
------------------------------------------------------------
         L0 MLP |        -4.303 | ▼ ████████████████████
        L10 MLP |        -2.189 | ▼ ██████████
    L11 Attn H0 |        -1.899 | ▼ ████████
    L0 Attn H11 |        +1.568 | ▲ ███████
   L11 Attn H10 |        +1.394 | ▲ ██████
     L0 Attn H8 |        +1.229 | ▲ █████
     L0 Attn H9 |        -1.189 | ▼ █████
     L5 Attn H9 |        -0.956 | ▼ ████
    L10 Attn H7 |        +0.784 | ▲ ███
         L8 MLP |        -0.741 | ▼ ███
     L5 Attn H1 |        -0.587 | ▼ ██
     L0 Attn H6 |        +0.554 | ▲ ██
    L10 Attn H0 |        -0.548 | ▼ ██
    L10 Attn H2 |        -0.546 | ▼ ██
         L5 MLP |        -0.506 | ▼ ██
     L1 Attn H7 |        -0.499 | ▼ ██
     L9 Attn H3 |        -0.493 | ▼ ██
    L0 Attn H

In [ ]:
# [Cell 12] Same as Cell 10 but NO HOOKS, NO EINSUM — just matmul and loops
# Every operation is a simple 2D matrix multiply you can follow

TARGET_FEATURE = 10261
prompt = "The Golden Gate Bridge spans"
tokens = model.to_tokens(prompt)  # [1, seq_len] integer token IDs

def manual_forward(tokens, ablate_type=None, ablate_layer=None, ablate_head=None):
    x = model.embed(tokens) + model.pos_embed(tokens)  # [1, seq_len, 768]
    n_heads = 12
    d_head = 64  # 768 / 12 = 64

    for layer_idx in range(8):
        block = model.blocks[layer_idx]
        normed = block.ln1(x)  # layer norm: [1, seq_len, 768]
        sl = normed.shape[1]   # seq_len

        if ablate_type == 'head' and ablate_layer == layer_idx:
            # --- MANUAL ATTENTION with plain matmul ---
            W_Q, W_K, W_V = block.attn.W_Q, block.attn.W_K, block.attn.W_V
            W_O = block.attn.W_O
            b_Q, b_K, b_V = block.attn.b_Q, block.attn.b_K, block.attn.b_V

            # Q, K, V per head: loop over 12 heads, each is a [seq, 768] @ [768, 64] = [seq, 64]
            q = torch.zeros(sl, n_heads, d_head, device=x.device)
            k = torch.zeros(sl, n_heads, d_head, device=x.device)
            v = torch.zeros(sl, n_heads, d_head, device=x.device)
            for h in range(n_heads):
                q[:, h, :] = normed[0] @ W_Q[h] + b_Q[h]   # [seq,768] @ [768,64] = [seq,64]
                k[:, h, :] = normed[0] @ W_K[h] + b_K[h]
                v[:, h, :] = normed[0] @ W_V[h] + b_V[h]

            # Attention scores per head: Q @ K^T / sqrt(64)
            scores = torch.zeros(n_heads, sl, sl, device=x.device)
            for h in range(n_heads):
                scores[h] = (q[:, h, :] @ k[:, h, :].T) / (d_head ** 0.5)  # [seq,64] @ [64,seq] = [seq,seq]

            # Causal mask: upper triangle = -inf, softmax zeros them out
            mask = torch.triu(torch.ones(sl, sl, device=x.device), diagonal=1).bool()
            scores.masked_fill_(mask, float('-inf'))
            pattern = torch.softmax(scores, dim=-1)  # [12, seq, seq]

            # Weighted sum of values: pattern @ V per head
            z = torch.zeros(sl, n_heads, d_head, device=x.device)
            for h in range(n_heads):
                z[:, h, :] = pattern[h] @ v[:, h, :]  # [seq,seq] @ [seq,64] = [seq,64]

            # >>> ABLATE: zero one head <<<
            z[:, ablate_head, :] = 0

            # Project each head back to 768 and sum all heads
            attn_out = torch.zeros(1, sl, 768, device=x.device)
            for h in range(n_heads):
                attn_out[0] += z[:, h, :] @ W_O[h]  # [seq,64] @ [64,768] = [seq,768]
            attn_out += block.attn.b_O
        else:
            attn_out = block.attn(normed)

        x = x + attn_out  # residual: add attention to stream

        # --- MLP ---
        normed2 = block.ln2(x)
        mlp_out = block.mlp(normed2)  # 768 -> 3072 -> 768

        if ablate_type == 'mlp' and ablate_layer == layer_idx:
            mlp_out = torch.zeros_like(mlp_out)  # >>> ABLATE <<<

        x = x + mlp_out  # residual: add MLP to stream

    return x

# --- Run ---
with torch.no_grad():
    baseline_resid = manual_forward(tokens)
    baseline_acts = torch.relu((baseline_resid - b_dec) @ W_enc + b_enc)

    token_strs = model.to_str_tokens(prompt)
    token_acts = baseline_acts[0, :, TARGET_FEATURE]
    best_tok = token_acts.argmax().item()
    baseline_val = token_acts[best_tok].item()

    print(f"Feature {TARGET_FEATURE} on '{token_strs[best_tok]}': {baseline_val:.3f}\n")

    results = []
    for layer in range(8):
        for head in range(12):
            resid = manual_forward(tokens, 'head', layer, head)
            acts = torch.relu((resid - b_dec) @ W_enc + b_enc)
            change = acts[0, best_tok, TARGET_FEATURE].item() - baseline_val
            results.append((f"L{layer} Attn H{head}", change))

        resid = manual_forward(tokens, 'mlp', layer)
        acts = torch.relu((resid - b_dec) @ W_enc + b_enc)
        change = acts[0, best_tok, TARGET_FEATURE].item() - baseline_val
        results.append((f"L{layer} MLP", change))

results.sort(key=lambda r: abs(r[1]), reverse=True)
max_change = max(abs(r[1]) for r in results) if results else 1

print(f"{'Component':>15} | {'Change':>10} | Impact")
print("-" * 55)
for name, change in results[:20]:
    if abs(change) < 0.001:
        continue
    bar = "█" * int(abs(change) / max_change * 20) if max_change > 0 else ""
    direction = "▼" if change < 0 else "▲"
    print(f"  {name:>13} | {change:>+10.3f} | {direction} {bar}")

In [ ]:
# [Cell 13] FULLY RAW circuit tracing — zero TransformerLens
# Only: torch (math), transformers (weight download), tiktoken (tokenizer)
# Every operation is visible. Nothing hidden.

import torch
import math
import tiktoken
from transformers import GPT2Model

# --- Load raw weights as flat dict of tensors ---
sd = GPT2Model.from_pretrained("gpt2").state_dict()
enc = tiktoken.get_encoding("gpt2")

# Print all weight names and shapes so you can see exactly what GPT-2 is
print("GPT-2 small weights:")
for name, param in sd.items():
    print(f"  {name:40s} {str(list(param.shape)):>20s}")

# --- Helper: layer norm from raw weights ---
def layer_norm(x, weight, bias, eps=1e-5):
    # x = input, weight = learned scale, bias = learned shift
    mean = x.mean(-1, keepdim=True)
    var = ((x - mean) ** 2).mean(-1, keepdim=True)
    return (x - mean) / torch.sqrt(var + eps) * weight + bias

# --- Helper: GELU activation (GPT-2 uses tanh approximation) ---
def gelu(x):
    return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x ** 3)))

# --- Tokenize (prepend BOS=50256 to match TransformerLens) ---
prompt = "The Golden Gate Bridge spans"
token_ids = [50256] + enc.encode(prompt)  # BOS + prompt tokens
tokens = torch.tensor([token_ids])        # [1, seq_len]
token_strs = [enc.decode([t]) for t in token_ids]
print(f"\nTokens: {token_strs}")
print(f"Token IDs: {token_ids}")

# --- Raw forward pass ---
def raw_forward(tokens, ablate_type=None, ablate_layer=None, ablate_head=None):
    seq_len = tokens.shape[1]
    n_heads, d_head = 12, 64  # 768 / 12 = 64

    # Embed: token embedding + position embedding
    x = sd['wte.weight'][tokens[0]]               # [seq, 768] — lookup rows by token ID
    x = x + sd['wpe.weight'][:seq_len]             # [seq, 768] — add position 0,1,2,...
    x = x.unsqueeze(0)                             # [1, seq, 768]

    for layer in range(8):  # layers 0-7 (SAE sits at layer 8 input)

        # === ATTENTION ===
        ln1_w = sd[f'h.{layer}.ln_1.weight']       # [768]
        ln1_b = sd[f'h.{layer}.ln_1.bias']         # [768]
        ln1_out = layer_norm(x, ln1_w, ln1_b)      # [1, seq, 768]

        # QKV projection: [1, seq, 768] @ [768, 2304] = [1, seq, 2304]
        W_qkv = sd[f'h.{layer}.attn.c_attn.weight']  # [768, 2304] — Q,K,V packed side by side
        b_qkv = sd[f'h.{layer}.attn.c_attn.bias']    # [2304]
        qkv = ln1_out @ W_qkv + b_qkv                # [1, seq, 2304]

        # Split into Q, K, V: each [1, seq, 768]
        q, k, v = qkv.split(768, dim=-1)

        # Reshape to per-head: [1, seq, 768] -> [1, seq, 12, 64] -> [1, 12, seq, 64]
        q = q.view(1, seq_len, n_heads, d_head).transpose(1, 2)
        k = k.view(1, seq_len, n_heads, d_head).transpose(1, 2)
        v = v.view(1, seq_len, n_heads, d_head).transpose(1, 2)

        # Attention scores: Q @ K^T / sqrt(64)
        # [1, 12, seq, 64] @ [1, 12, 64, seq] = [1, 12, seq, seq]
        scores = (q @ k.transpose(-2, -1)) / (d_head ** 0.5)

        # Causal mask: can't look at future tokens
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        scores.masked_fill_(mask, float('-inf'))
        pattern = torch.softmax(scores, dim=-1)  # [1, 12, seq, seq]

        # Weighted values: [1, 12, seq, seq] @ [1, 12, seq, 64] = [1, 12, seq, 64]
        z = pattern @ v

        # >>> ABLATE: zero one head <<<
        if ablate_type == 'head' and ablate_layer == layer:
            z[0, ablate_head, :, :] = 0

        # Merge heads back: [1, 12, seq, 64] -> [1, seq, 768]
        z = z.transpose(1, 2).contiguous().view(1, seq_len, 768)

        # Output projection: [1, seq, 768] @ [768, 768] = [1, seq, 768]
        W_proj = sd[f'h.{layer}.attn.c_proj.weight']  # [768, 768]
        b_proj = sd[f'h.{layer}.attn.c_proj.bias']    # [768]
        attn_out = z @ W_proj + b_proj

        x = x + attn_out  # residual connection

        # === MLP ===
        ln2_w = sd[f'h.{layer}.ln_2.weight']       # [768]
        ln2_b = sd[f'h.{layer}.ln_2.bias']         # [768]
        ln2_out = layer_norm(x, ln2_w, ln2_b)      # [1, seq, 768]

        # Up projection: [1, seq, 768] @ [768, 3072] = [1, seq, 3072]
        W_fc = sd[f'h.{layer}.mlp.c_fc.weight']    # [768, 3072]
        b_fc = sd[f'h.{layer}.mlp.c_fc.bias']      # [3072]
        hidden = gelu(ln2_out @ W_fc + b_fc)       # [1, seq, 3072]

        # Down projection: [1, seq, 3072] @ [3072, 768] = [1, seq, 768]
        W_down = sd[f'h.{layer}.mlp.c_proj.weight'] # [3072, 768]
        b_down = sd[f'h.{layer}.mlp.c_proj.bias']   # [768]
        mlp_out = hidden @ W_down + b_down           # [1, seq, 768]

        # >>> ABLATE: zero MLP <<<
        if ablate_type == 'mlp' and ablate_layer == layer:
            mlp_out = torch.zeros_like(mlp_out)

        x = x + mlp_out  # residual connection

    return x  # [1, seq, 768] — residual stream at layer 8

# --- Circuit trace ---
TARGET_FEATURE = 10261

with torch.no_grad():
    baseline_resid = raw_forward(tokens)
    baseline_acts = torch.relu((baseline_resid - b_dec) @ W_enc + b_enc)

    token_acts = baseline_acts[0, :, TARGET_FEATURE]
    best_tok = token_acts.argmax().item()
    baseline_val = token_acts[best_tok].item()

    print(f"\nFeature {TARGET_FEATURE} on '{token_strs[best_tok]}': {baseline_val:.3f}")
    print("(Compare to Cell 10/12 — should be same value)\n")

    results = []
    for layer in range(8):
        for head in range(12):
            resid = raw_forward(tokens, 'head', layer, head)
            acts = torch.relu((resid - b_dec) @ W_enc + b_enc)
            change = acts[0, best_tok, TARGET_FEATURE].item() - baseline_val
            results.append((f"L{layer} Attn H{head}", change))

        resid = raw_forward(tokens, 'mlp', layer)
        acts = torch.relu((resid - b_dec) @ W_enc + b_enc)
        change = acts[0, best_tok, TARGET_FEATURE].item() - baseline_val
        results.append((f"L{layer} MLP", change))

results.sort(key=lambda r: abs(r[1]), reverse=True)
max_change = max(abs(r[1]) for r in results) if results else 1

print(f"{'Component':>15} | {'Change':>10} | Impact")
print("-" * 55)
for name, change in results[:20]:
    if abs(change) < 0.001:
        continue
    bar = "█" * int(abs(change) / max_change * 20) if max_change > 0 else ""
    direction = "▼" if change < 0 else "▲"
    print(f"  {name:>13} | {change:>+10.3f} | {direction} {bar}")